In [0]:
%sql
USE e_commerce.bronze;

In [0]:
%run "../src/utils/common_functions"

In [0]:
%run "../src/utils/configuration"

In [0]:
%run "../src/utils/merge"

In [0]:
df_customer = spark.read.format("delta").table("bronze.customer")

In [0]:
reglas = []

In [0]:
from pyspark.sql.functions import col, current_timestamp
import json

In [0]:
nulos(df_customer, "customer_id", reglas, "customer")
duplicados(df_customer, "customer_id", reglas, "customer")

In [0]:
df_reglas = spark.createDataFrame(reglas)
df_reglas = df_reglas.withColumn("validation_date", current_timestamp())

In [0]:
df_reglas.write.mode("append").format("delta").saveAsTable("bronze.log_transformation")

In [0]:
error = [regla for regla in reglas if not regla["cumple"]]

if error:
    dbutils.jobs.taskValues.set(key="estado", value="Error crítico")
    dbutils.jobs.taskValues.set(key="detalle", value=json.dumps(error))
else:
    dbutils.jobs.taskValues.set(key="estado", value="Validación exitosa")

In [0]:
%sql
SELECT * FROM bronze.log_transformation;

In [0]:
last_record = get_last_record(spark, "e_commerce.silver.customer")

df_customer_incremental = spark.read.table("e_commerce.bronze.customer") \
    .filter(f"ingestion_timestamp > '{last_record}'")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("customer_id") \
               .orderBy(col("ingestion_timestamp").desc())

df_customer_silver = df_customer_incremental \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

In [0]:
df_customer_silver = df_customer_silver\
    .withColumnRenamed("customer_zip_code_prefix", "zip_code_prefix")\
    .withColumnRenamed("customer_city", "city")\
    .withColumnRenamed("customer_state", "state")

In [0]:
merge(
    df_source=df_customer_silver,
    target_table="e_commerce.silver.customer",
    merge_key="customer_id",
    transforms={
        "city": "INITCAP(s.city)"
    },
    exclude_update_cols=["source_name"]  # opcional
)

In [0]:
%sql
SELECT file_date, COUNT(1) FROM silver.customer GROUP BY file_date;

In [0]:
%sql
SELECT * FROM bronze.log_transformation WHERE tabla = 'customer' ORDER BY validation_date DESC;

In [0]:
%sql
SELECT * FROM silver.customer;